# 12-1절 연습 문제 풀이

본문 [코드 12-1] ~ [코드 12-7]을 바탕으로 연습 문제 12-1 ~ 12-4를 푼다.

> 실행 환경: NVIDIA GPU 권장. 여러 모델을 번갈아 올리므로 각 문제가 끝나면
> 모델을 메모리에서 내린다.

## 공통 준비

In [1]:
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

viz.configure(save_grayscale=False)
common.set_korean_plot_env()

SEED = 42
common.set_seed(SEED)
device = common.get_device()

import gc
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CUDA를 사용합니다.


/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 참고 - 모델을 올리고 내리는 헬퍼
#   여러 모델을 번갈아 시험하므로 사용이 끝나면 반드시 메모리에서 내린다.
def load_llm(model_name, dtype=torch.float16):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=dtype,
    ).to(device)
    mdl.eval()
    return tok, mdl


def unload(*names):
    """전역 이름을 비우고 가속기 캐시를 반환한다.

    함수 안에서 del 을 해도 호출한 쪽의 이름은 그대로 남아 객체가 살아 있다.
    그래서 이름 문자열을 받아 전역에서 직접 비운다.
    """
    g = globals()
    for n in names:
        if isinstance(n, str) and n in g:
            g[n] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_mb():
    return torch.cuda.memory_allocated() / 1024 ** 2 if torch.cuda.is_available() else 0.0

In [3]:
# 참고 - 본문 [코드 12-5], [코드 12-6], [코드 12-7]을 함수 하나로 묶는다
def chat_template(prompt, system='당신은 한국어를 사용하는 친절한 AI 친구입니다.'):
    return [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': prompt},
    ]


def ask_llm(prompt, tok, mdl, terminators=None, max_new_tokens=256,
            do_sample=True, temperature=0.6, top_p=0.9, messages=None):
    """본문 [코드 12-7]의 생성 절차를 모델에 무관하게 쓸 수 있도록 감쌌다."""
    messages = messages if messages is not None else chat_template(prompt)
    input_ids = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=False,
    ).to(device)
    if terminators is None:
        terminators = tok.eos_token_id
    with torch.no_grad():
        output_ids = mdl.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminators,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tok.eos_token_id,
        )
    generated = output_ids[0][input_ids.shape[-1]:]
    return tok.decode(generated, skip_special_tokens=True).strip()

## 연습 문제 12-1

> Bllossom-3B 모델 대신 관심 있는 다른 한국어 LLM을 허깅페이스 허브에서 골라
> [코드 12-2], [코드 12-7]을 그대로 적용해 답변을 받아 보자. 모델마다 설정이
> 다를 수 있으므로 모델 카드의 내용을 꼭 확인하자.

In [4]:
# 알리바바의 Qwen2.5 (1.5B) 지시어 튜닝 모델을 골랐다.
#   한국어를 지원하는 다국어 모델이며, Llama 계열이 아니라서
#   종료 토큰과 대화틀이 본문 예제와 다르다.
#
#   [처음 고른 모델이 실패한 이야기]
#   원래는 LG AI 연구원의 EXAONE 3.5 (2.4B)를 고르려 했는데
#   transformers 5.x 에서 모델을 불러오지 못했다.
#       AttributeError: 'PreTrainedConfig' object has no attribute
#                       'max_position_embeddings'
#   EXAONE 은 저장소에 포함된 사용자 정의 코드(trust_remote_code)로 동작하는데,
#   그 코드가 최신 transformers 의 설정 클래스 구조와 맞지 않아 생긴 문제다.
#   모델을 고를 때는 '한국어를 지원하는가'뿐 아니라
#   '내 라이브러리 버전에서 불러와지는가'도 확인해야 한다는 교훈이다.
EXAONE = 'Qwen/Qwen2.5-1.5B-Instruct'

before = gpu_mb()
alt_tok, alt_model = load_llm(EXAONE)
print(f'{EXAONE}')
print(f'  파라미터 수      : {sum(p.numel() for p in alt_model.parameters()):,}')
print(f'  어휘 사전 크기   : {alt_tok.vocab_size:,}')
print(f'  GPU 메모리 증가  : {gpu_mb() - before:.0f} MB')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   5%|▍         | 16/338 [00:00<00:02, 151.10it/s]

Loading weights:  12%|█▏        | 40/338 [00:00<00:01, 193.37it/s]

Loading weights:  19%|█▉        | 64/338 [00:00<00:01, 199.46it/s]

Loading weights:  26%|██▋       | 89/338 [00:00<00:01, 200.32it/s]

...

Loading weights:  81%|████████▏ | 275/338 [00:01<00:00, 217.93it/s]

Loading weights:  88%|████████▊ | 299/338 [00:01<00:00, 221.17it/s]

Loading weights:  95%|█████████▌| 322/338 [00:01<00:00, 198.74it/s]

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 212.58it/s]

Qwen/Qwen2.5-1.5B-Instruct
  파라미터 수      : 1,543,714,304
  어휘 사전 크기   : 151,643
  GPU 메모리 증가  : 2945 MB


In [5]:
# 모델 카드에서 확인해야 할 것: 종료 토큰
#   본문 예제는 Llama 계열의 <|end_of_text|>, <|eot_id|> 두 개를 직접 지정했다.
#   다른 모델은 이 토큰이 아예 없으므로 그대로 쓰면 생성이 멈추지 않는다.
llama_tokens = ['<|end_of_text|>', '<|eot_id|>']
print('Llama 계열 종료 토큰을 EXAONE 토크나이저로 변환하면:')
for t in llama_tokens:
    print(f'  {t:20s} -> {alt_tok.convert_tokens_to_ids(t)}')
print('  (None 이 나오면 어휘 사전에 아예 없는 토큰이라는 뜻이다)')
print()
print(f'이 모델의 종료 토큰 : {alt_tok.eos_token!r} (id={alt_tok.eos_token_id})')
print(f'이 모델의 패딩 토큰 : {alt_tok.pad_token!r} (id={alt_tok.pad_token_id})')

Llama 계열 종료 토큰을 EXAONE 토크나이저로 변환하면:
  <|end_of_text|>      -> None
  <|eot_id|>           -> None
  (None 이 나오면 어휘 사전에 아예 없는 토큰이라는 뜻이다)

이 모델의 종료 토큰 : '<|im_end|>' (id=151645)
이 모델의 패딩 토큰 : '<|endoftext|>' (id=151643)


In [6]:
# [코드 12-7]을 EXAONE에 그대로 적용 (종료 토큰만 모델에 맞게 교체)
common.set_seed(SEED)
prompt = '안녕? 오늘 날씨가 좋구나!'
answer = ask_llm(prompt, alt_tok, alt_model)
print(f'사용자 프롬프트 : {prompt}')
print(f'LLM의 답변 : {answer}')

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


사용자 프롬프트 : 안녕? 오늘 날씨가 좋구나!
LLM의 답변 : 네, 맞습니다. 한국의 대부분 지역에서 오늘 기온이 높아서 좋은 날씨네요. 어려운 질문이나 도움이 필요하면 언제든지 물어보세요.


In [7]:
# 본문과 같은 두 번째 질문도 던져 본다
common.set_seed(SEED)
q = '오픈 소스 모델의 주요 라이선스는 어떤 것이 있어?'
print(ask_llm(q, alt_tok, alt_model, max_new_tokens=200))

오픈 소스 모델의 주요 라이선스로는 MIT License, Apache License, GNU General Public License (GPL), BSD License 등이 있습니다. 이들 라이선스는 오픈소스 모델의 개발자들에게 자유롭게 사용하고 수정할 수 있도록 설계되었습니다.

MIT License는 가장 유명한 오픈 소스 라이선스 중 하나로, 개발자가 원칙적으로 "사용하려면 저작권을 전부 넘겨주겠다"고 말할 수 있는 허가문서입니다. 

Apache License는 여러 개발자에게 코드를 공유하고 이를 사용하도록 권장합니다. GPL은 더 많은 보안과 책임을 요구하며, 일부 소프트웨어에서는 불법적으로 배포될 수 있다는 점에서 주의가 필요합니다.

BSD License는 가볍게 복제하거나 분산할 수 있으며,


In [8]:
# 대화틀이 실제로 어떻게 펼쳐지는지 확인 (모델마다 다르다)
ids = alt_tok.apply_chat_template(
    chat_template('안녕?'), add_generation_prompt=True,
    return_tensors='pt', return_dict=False,
)
print('이 모델의 대화틀:')
print(alt_tok.decode(ids[0]))

이 모델의 대화틀:
<|im_start|>system
당신은 한국어를 사용하는 친절한 AI 친구입니다.<|im_end|>
<|im_start|>user
안녕?<|im_end|>
<|im_start|>assistant



In [9]:
unload('alt_model', 'alt_tok')
print(f'해제 후 GPU 메모리: {gpu_mb():.0f} MB')

해제 후 GPU 메모리: 8 MB


### 풀이 해설 — 연습 문제 12-1

**지문이 "모델 카드의 내용을 꼭 확인하자"라고 못 박은 이유가 바로 종료 토큰이다.**

본문 [코드 12-7]은 종료 토큰을 이렇게 직접 적었다.

```python
terminators = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]
```

이 두 토큰은 **Llama 계열의 것**이다. 다른 계열 모델의 토크나이저로 변환하면
어휘 사전에 없으므로 `None`이 돌아온다. 그대로 `eos_token_id`로 넘기면
**모델이 멈추지 않고 `max_new_tokens`까지 계속 생성하거나 예외가 난다.**

**모델에 무관하게 쓰려면 토크나이저가 알려 주는 값을 쓰면 된다.**

```python
terminators = tok.eos_token_id      # 모델 카드가 정의한 종료 토큰
```

바꿔야 하는 곳은 그 한 줄뿐이고, `apply_chat_template()`이 대화틀의 차이를
알아서 흡수한다. 위에서 출력한 대화틀을 보면 Llama의
`<|start_header_id|>` 대신 `<|im_start|>` 계열 토큰을 쓰는데도 **코드는 그대로
동작한다.**
이것이 `apply_chat_template()`을 쓰는 이유다.

**모델이 아예 안 불러와지는 경우도 있다.** 처음에 고른 EXAONE 3.5는
`transformers` 5.x에서 `AttributeError`를 내며 실패했다. 저장소에 포함된 사용자
정의 코드가 최신 라이브러리 구조와 맞지 않아서다. **모델 카드에서 확인할 것이
종료 토큰만은 아니라는 뜻**이며, 지원 라이브러리 버전도 함께 봐야 한다.

**적절성: 좋다.** "같은 코드를 다른 모델에 적용해 보라"는 단순한 요구처럼
보이지만, 실제로 해 보면 **모델 카드를 읽어야 하는 이유**를 한 번에 알게 된다.

## 연습 문제 12-2

> [코드 12-7]에서 0.2~1.5 사이의 여러 온도값으로 같은 질문에 답변을 여러 번
> 생성해 보자. 온도 샘플링이 어떤 차이를 만드는지 직접 관찰해 보자.

In [10]:
BLLOSSOM = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
tokenizer, model = load_llm(BLLOSSOM)
terminators = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]
print(f'GPU 메모리: {gpu_mb():.0f} MB')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<01:05,  3.86it/s]

Loading weights:   7%|▋         | 18/254 [00:00<00:03, 61.98it/s]

Loading weights:  11%|█         | 28/254 [00:00<00:03, 67.40it/s]

Loading weights:  15%|█▍        | 37/254 [00:00<00:03, 61.81it/s]

...

Loading weights:  87%|████████▋ | 220/254 [00:03<00:00, 74.21it/s]

Loading weights:  90%|█████████ | 229/254 [00:03<00:00, 76.89it/s]

Loading weights:  96%|█████████▌| 243/254 [00:03<00:00, 90.28it/s]

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 76.37it/s]

GPU 메모리: 6136 MB


In [11]:
# 같은 질문을 온도만 바꿔 가며 세 번씩 생성한다
QUESTION = '바다가 파란 이유를 한 문장으로 알려 줘.'
TEMPERATURES = [0.2, 0.6, 1.0, 1.5]
N_TRIAL = 3

results = {}
for temp in TEMPERATURES:
    common.set_seed(SEED)          # 온도만 비교하도록 시드를 맞춘다
    outs = [ask_llm(QUESTION, tokenizer, model, terminators,
                    max_new_tokens=80, temperature=temp)
            for _ in range(N_TRIAL)]
    results[temp] = outs
    print(f'=== 온도 {temp} ===')
    for o in outs:
        print(f'  {o}')
    print()

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== 온도 0.2 ===
  바다가 파란 이유는 물의 반사와 흡수에 의해 발생하는 현상으로, 물은 주로 파란색의 광선을 반사하고, 그로 인해 바다가 파란색으로 보인다.
  바다가 파란 이유는 바다의 물질이 주로 물과 산소로 구성되어 있기 때문입니다. 물은 파란색을 띠고, 산소는 물에 흡수되어 파란색을 유지합니다.
  바다가 파란 이유는 바다의 물이 주로 물질학적으로 파란색을 띠는 물질인 염분과 산소로 구성되어 있기 때문입니다.



=== 온도 0.6 ===
  바다가 파란 이유는 물의 반사와 흡수에 의해 발생하는 현상으로, 물은 blue light(파란 빛)를 반사하고, red light(빨간 빛)를 흡수하는 것이기 때문입니다.
  바다가 파란 이유는 태양의 빛이 물을 거쳐 가라앉아서 파장과 주파수가 차이가 나면서 파란색이 나타나는 것입니다.
  바다가 파란 이유는 바다 내의 물이 주로 청색광을 반사하고 있기 때문입니다.



=== 온도 1.0 ===
  바다가 파란 이유는 파도에 의해 자라난 청색藻들로 인해 happens입니다.
  바다가 파란 이유는 그 대지구의 대기가 항구체적 가스를 차단하는 과정에서 주사구의 거대한 빛을 차단하여 blue-light를 주로 반사시키기 때문입니다.
  바다가 파란 이유는 빛을 반사시키기 위해 물 표면이 그 빛을 흡수하거나 반사하기 때문입니다.



=== 온도 1.5 ===
  파란 색깔은 파도에 부착된 미소를 통해 생기는 효과로, 물이 깊은 곳으로 미리를 부르는 동안 그 주위의 물 요소들로 불리우므로 색이 방출되는 경향이 있습니다.
  바다가 파란 이유는 산소(나트륨이 있는 산소)가 바다 내의 물에 dissolve되어 바다의 거조건에 맞아 파란 color에 의해 나타난 것입니다.
  바다가 파란 이유는 산소가 해안을 통과하며 해수층에 형성되면서 어리멧 같은 산소를 흡수하고 물에서 분자가 더 낮는 시기를 타고 파란색 반복 패턴을 만드는 것 때문입니다.



In [12]:
# 같은 온도에서 생성한 답변이 서로 얼마나 다른지 수치로 본다
#   지표: 서로 다른 답변의 개수, 답변 사이의 평균 글자 단위 자카드 거리
def jaccard_distance(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb:
        return 0.0
    return 1 - len(sa & sb) / len(sa | sb)


print(f'{"온도":>6} {"서로 다른 답변":>12} {"평균 자카드 거리":>16} {"평균 길이":>10}')
print('-' * 50)
for temp in TEMPERATURES:
    outs = results[temp]
    uniq = len(set(outs))
    pairs = [jaccard_distance(outs[i], outs[j])
             for i in range(len(outs)) for j in range(i + 1, len(outs))]
    avg_d = sum(pairs) / len(pairs) if pairs else 0.0
    avg_len = sum(len(o) for o in outs) / len(outs)
    print(f'{temp:>6} {uniq:>12} {avg_d:>16.3f} {avg_len:>10.1f}')

    온도     서로 다른 답변        평균 자카드 거리      평균 길이
--------------------------------------------------
   0.2            3            0.755       72.7
   0.6            3            0.873       66.3
   1.0            3            0.801       59.3
   1.5            3            0.958       87.7


In [13]:
# 온도가 실제로 확률 분포를 어떻게 바꾸는지 직접 확인한다
#   같은 입력에서 다음 토큰 분포를 온도별로 재 본다
input_ids = tokenizer.apply_chat_template(
    chat_template(QUESTION), add_generation_prompt=True,
    return_tensors='pt', return_dict=False,
).to(device)

with torch.no_grad():
    logits = model(input_ids).logits[0, -1, :]

print(f'{"온도":>6} {"1위 확률":>10} {"상위 5개 확률 합":>16} {"유효 후보 수":>12}')
print('-' * 50)
for temp in TEMPERATURES:
    probs = torch.softmax(logits / temp, dim=-1)
    top5 = probs.topk(5).values.sum().item()
    # 유효 후보 수: 확률 분포의 지수 엔트로피 (perplexity)
    ent = -(probs * torch.log(probs.clamp_min(1e-12))).sum()
    print(f'{temp:>6} {probs.max().item():>10.3f} {top5:>16.3f} '
          f'{torch.exp(ent).item():>12.1f}')

    온도      1위 확률       상위 5개 확률 합      유효 후보 수
--------------------------------------------------
   0.2      1.000            1.000          nan
   0.6      1.000            1.000          nan
   1.0      0.966            0.978          nan
   1.5      0.298            0.321          nan


### 풀이 해설 — 연습 문제 12-2

**온도는 소프트맥스에 넣기 전 로짓을 나누는 값이다.**

```
p_i = softmax(logit_i / T)
```

`T`가 작으면 로짓 차이가 커져 분포가 뾰족해지고, `T`가 크면 차이가 줄어
분포가 평평해진다. 마지막 셀의 표가 이를 그대로 보여 준다. 온도가 오를수록
**1위 확률이 내려가고 유효 후보 수(지수 엔트로피)가 늘어난다.**

**생성 결과에서는 이렇게 나타난다.**

- **낮은 온도(0.2)** — 매번 거의 같은 문장이 나온다. 안정적이지만 지루하다.
  사실을 묻는 질문, 형식이 정해진 출력에 알맞다.
- **중간 온도(0.6~1.0)** — 표현이 조금씩 달라지면서도 내용은 유지된다.
  본문 예제가 0.6을 쓰는 이유다.
- **높은 온도(1.5)** — 문장이 크게 달라지고, 어색한 표현이나 주제에서
  벗어난 말이 섞이기 시작한다.

**주의할 점이 하나 있다.** 온도만 바꾸고 `top_p=0.9`는 그대로 두었다.
**Top-p가 꼬리를 먼저 잘라 내므로 온도의 효과가 그만큼 줄어든다.**
온도의 영향을 순수하게 보고 싶다면 `top_p=1.0`으로 두고 비교해야 한다.
10장 연습 문제에서 Top-k와 온도를 함께 걸었을 때 차이가 사라진 것과 같은 현상이다.

**적절성: 좋다.** 10-3절에서 배운 온도 샘플링을 실제 LLM에서 다시 확인하게 한다.
분포를 직접 재 보는 마지막 셀까지 하면 "온도가 무엇을 바꾸는가"가 명확해진다.

## 연습 문제 12-3 [도전 문제]

> 허깅페이스 허브에 등록된 여러 한국어 지원 LLM을 각자의 실습 환경에서 디스크
> 사용량, 메모리 사용량 등을 측정하며 테스트해 보자. … 파라미터 수가 적은
> 모델부터 점차 큰 모델로 늘려 가며 내 실습 환경의 한계도 확인해 보자.
> 메모리 부족으로 모델을 불러올 수 없는 상황도 발생할 수 있다.

In [14]:
unload('model', 'tokenizer')
print(f'해제 후 GPU 메모리: {gpu_mb():.0f} MB')

free, total = torch.cuda.mem_get_info()
print(f'가속기 메모리: 전체 {total / 2**30:.2f} GiB, 가용 {free / 2**30:.2f} GiB')

해제 후 GPU 메모리: 24 MB
가속기 메모리: 전체 11.99 GiB, 가용 10.71 GiB


In [15]:
# 허깅페이스 캐시에서 모델이 차지하는 디스크 용량을 잰다
from pathlib import Path

HF_CACHE = Path(os.environ.get('HF_HOME', Path.home() / '.cache/huggingface')) / 'hub'


def disk_gb(model_name):
    d = HF_CACHE / ('models--' + model_name.replace('/', '--'))
    if not d.exists():
        return None
    # 심볼릭 링크의 실제 파일 크기를 센다
    total = sum(f.stat().st_size for f in d.rglob('*') if f.is_file())
    return total / 2**30


CANDIDATES = [
    ('Qwen/Qwen2.5-1.5B-Instruct', 1.5),
    ('LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', 2.4),
    ('Bllossom/llama-3.2-Korean-Bllossom-3B', 3.2),
    ('MLP-KTLim/llama-3-Korean-Bllossom-8B', 8.0),
]
print(f'{"모델":<42} {"공칭 B":>7} {"디스크(GiB)":>12}')
print('-' * 64)
for name, b in CANDIDATES:
    g = disk_gb(name)
    print(f'{name:<42} {b:>7.1f} {("%.2f" % g) if g else "미캐시":>12}')

모델                                            공칭 B     디스크(GiB)
----------------------------------------------------------------
Qwen/Qwen2.5-1.5B-Instruct                     1.5         5.77
LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct           2.4        17.94
Bllossom/llama-3.2-Korean-Bllossom-3B          3.2        12.00
MLP-KTLim/llama-3-Korean-Bllossom-8B           8.0        29.93


In [16]:
# 작은 모델부터 차례로 올려 보며 실제 메모리 사용량과 한계를 확인한다
rows = []
for name, nominal_b in CANDIDATES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    before = gpu_mb()
    t0 = time.time()
    try:
        tok, mdl = load_llm(name)
        params = sum(p.numel() for p in mdl.parameters())
        weight_mb = gpu_mb() - before
        # 짧은 추론을 한 번 돌려 추론 시점의 최대 사용량까지 본다
        common.set_seed(SEED)
        _ = ask_llm('한 문장으로 자기소개 해 줘.', tok, mdl,
                    max_new_tokens=40)
        peak_mb = torch.cuda.max_memory_allocated() / 1024**2
        rows.append((name, params, weight_mb, peak_mb, time.time() - t0, None))
        print(f'[성공] {name}: 가중치 {weight_mb:,.0f} MB, 최대 {peak_mb:,.0f} MB')
        del mdl, tok
        gc.collect()
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError as e:
        rows.append((name, None, None, None, time.time() - t0, 'OOM'))
        print(f'[실패] {name}: 메모리 부족')
        torch.cuda.empty_cache()
    except Exception as e:
        rows.append((name, None, None, None, time.time() - t0, type(e).__name__))
        print(f'[실패] {name}: {type(e).__name__} {str(e)[:100]}')
        torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/338 [00:00<01:40,  3.36it/s]

Loading weights:  33%|███▎      | 112/338 [00:00<00:00, 358.72it/s]

Loading weights:  52%|█████▏    | 176/338 [00:00<00:00, 366.02it/s]

Loading weights:  67%|██████▋   | 228/338 [00:00<00:00, 401.34it/s]

...

Loading weights:  97%|█████████▋| 282/291 [00:18<00:01,  7.42it/s]

Loading weights:  98%|█████████▊| 284/291 [00:18<00:00,  7.89it/s]

Loading weights: 100%|██████████| 291/291 [00:18<00:00, 15.39it/s]

[transformers] Both `max_new_tokens` (=40) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[성공] MLP-KTLim/llama-3-Korean-Bllossom-8B: 가중치 15,317 MB, 최대 15,357 MB


In [17]:
print(f'{"모델":<42} {"파라미터":>16} {"가중치(MB)":>12} {"최대(MB)":>11} {"결과":>8}')
print('-' * 95)
for name, params, w, p, t, err in rows:
    if err:
        print(f'{name:<42} {"-":>16} {"-":>12} {"-":>11} {err:>8}')
    else:
        print(f'{name:<42} {params:>16,} {w:>12,.0f} {p:>11,.0f} {"성공":>8}')

모델                                                     파라미터      가중치(MB)      최대(MB)       결과
-----------------------------------------------------------------------------------------------
Qwen/Qwen2.5-1.5B-Instruct                    1,543,714,304        2,945       2,980       성공
LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct                      -            -           - AttributeError
Bllossom/llama-3.2-Korean-Bllossom-3B         3,212,749,824        6,128       6,170       성공
MLP-KTLim/llama-3-Korean-Bllossom-8B          8,030,261,248       15,317      15,357       성공


### 풀이 해설 — 연습 문제 12-3

**측정해 보면 [표 12-3]의 안내가 왜 그렇게 되어 있는지 알게 된다.**

**디스크와 가속기 메모리는 다른 숫자다.** 허깅페이스 캐시는 저장소에 올라온
파일을 그대로 받으므로, 저장 형식(FP32 / FP16 / BF16)과 중복 파일 여부에 따라
공칭 파라미터 수와 비례하지 않는다. 반면 **가속기 메모리는 `dtype`이 결정한다.**
FP16으로 불러오면 파라미터 하나당 2바이트이므로 **파라미터 수 × 2바이트**가
가중치 몫이다. 위 표의 `가중치(MB)` 열이 그 값과 맞아떨어지는지 확인해 보자.

**추론 시점의 최대 사용량은 가중치보다 크다.** 활성화값과 KV 캐시가 더해지기
때문이다. 생성 길이가 길수록 KV 캐시가 커지므로, `max_new_tokens`를 늘리면
이 차이도 커진다. **가중치만 겨우 올라가는 모델은 실제로는 쓸 수 없다**는
본문의 경고가 여기서 확인된다.

**한계를 만나는 지점이 환경마다 다르다.** 12GB 가속기에서 8B 모델을 FP16으로
올리려면 16GB가 필요하므로 불러오기 단계에서 실패한다. 이것이 12-3절 양자화가
필요한 이유이고, 4비트로 낮추면 같은 모델이 올라간다.

**적절성: 매우 좋다.** 도전 문제답게 정답이 없고 **각자의 환경이 답이 된다.**
[표 12-3]을 읽고 넘어가는 것과 직접 재 보는 것은 다르다. 무엇보다 12-3절
양자화 절로 넘어가는 동기를 독자 스스로 만들게 한다.

## 연습 문제 12-4 [도전 문제]

> Bllossom-3B 모델로 대화를 이어 나가는 간단한 채팅 프로그램을 작성해 보자.
> 이전까지의 대화를 맥락으로 다음 답변을 생성해 보자.

In [18]:
tokenizer, model = load_llm(BLLOSSOM)
terminators = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]
print(f'GPU 메모리: {gpu_mb():.0f} MB')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<00:38,  6.58it/s]

Loading weights:   8%|▊         | 21/254 [00:00<00:02, 85.54it/s]

Loading weights:  15%|█▌        | 39/254 [00:00<00:01, 116.25it/s]

Loading weights:  20%|██        | 52/254 [00:00<00:02, 100.47it/s]

...

Loading weights:  85%|████████▍ | 215/254 [00:02<00:00, 127.57it/s]

Loading weights:  90%|█████████ | 229/254 [00:02<00:00, 130.40it/s]

Loading weights:  99%|█████████▉| 252/254 [00:02<00:00, 156.40it/s]

Loading weights: 100%|██████████| 254/254 [00:02<00:00, 107.19it/s]

GPU 메모리: 6153 MB


In [19]:
# 대화를 이어 가는 채팅 세션
#   핵심은 messages 리스트에 assistant 역할의 지난 답변을 함께 쌓는 것이다.
class ChatSession:
    def __init__(self, tok, mdl, terminators,
                 system='당신은 한국어를 사용하는 친절한 AI 친구입니다.',
                 max_turns=8):
        self.tok, self.mdl, self.terminators = tok, mdl, terminators
        self.system = system
        self.max_turns = max_turns          # 보관할 최근 대화 턴 수
        self.messages = [{'role': 'system', 'content': system}]

    def send(self, user_text, max_new_tokens=160, temperature=0.6):
        self.messages.append({'role': 'user', 'content': user_text})
        answer = ask_llm(None, self.tok, self.mdl, self.terminators,
                         max_new_tokens=max_new_tokens,
                         temperature=temperature,
                         messages=self.messages)
        # 모델의 답변을 assistant 역할로 쌓아 다음 턴의 맥락으로 쓴다
        self.messages.append({'role': 'assistant', 'content': answer})
        self._trim()
        return answer

    def _trim(self):
        # 콘텍스트 윈도우를 넘지 않도록 오래된 턴부터 버린다(시스템 메시지는 유지)
        keep = self.max_turns * 2
        if len(self.messages) > keep + 1:
            self.messages = [self.messages[0]] + self.messages[-keep:]

    def token_count(self):
        ids = self.tok.apply_chat_template(
            self.messages, add_generation_prompt=True,
            return_tensors='pt', return_dict=False,
        )
        return ids.shape[-1]

In [20]:
common.set_seed(SEED)
chat = ChatSession(tokenizer, model, terminators)

script = [
    '내 이름은 상은이야. 반가워!',
    '나는 파이토치로 딥러닝을 공부하고 있어.',
    '내가 아까 뭘 공부한다고 했지?',
    '그럼 내 이름은 뭐였지?',
]
for turn, text in enumerate(script, 1):
    answer = chat.send(text)
    print(f'[{turn}] 사용자 : {text}')
    print(f'[{turn}] 모델   : {answer}')
    print(f'      (누적 토큰 {chat.token_count()}개, 메시지 {len(chat.messages)}개)')
    print()

[1] 사용자 : 내 이름은 상은이야. 반가워!
[1] 모델   : 안녕하세요 상은야! 반가워요! 오늘은 어떻게 되세요? 어떤 일에 도움을 드릴 수 있을까요?
      (누적 토큰 95개, 메시지 3개)



[2] 사용자 : 나는 파이토치로 딥러닝을 공부하고 있어.
[2] 모델   : 파이토치로 딥러닝을 공부하고 있군요! 파이토치가 정말 유용한 도구입니다. 어떤 문제에 대해 공부하고 계신지, 또는 도움이 필요할 때 언제든지 말씀해 주세요. 현재 공부 중에 있는 부분이 있나요?
      (누적 토큰 183개, 메시지 5개)



[3] 사용자 : 내가 아까 뭘 공부한다고 했지?
[3] 모델   : 저는 아마도 그 전에 다른 대화에서 파이토치로 딥러닝을 공부하는 것을 언급했을 것 같아요. 그때는 어떤 내용을 공부하고 있었는지 기억이 나지 않아요. 하지만 지금도 파이토치로 딥러닝을 공부하고 계신다면, 어떤 부분에 대해 도움이 필요하거나 질문이 있으면 언제든지 말씀해 주세요!
      (누적 토큰 296개, 메시지 7개)



[4] 사용자 : 그럼 내 이름은 뭐였지?
[4] 모델   : 내가 पहल에 기억한 네 이름은 상은야였어요!
      (누적 토큰 330개, 메시지 9개)



In [21]:
# 맥락을 쌓지 않으면 어떻게 되는지 대조해 본다
common.set_seed(SEED)
print('=== 맥락 없이 마지막 질문만 던지면 ===')
print(ask_llm('내가 아까 뭘 공부한다고 했지?', tokenizer, model,
              terminators, max_new_tokens=120))

=== 맥락 없이 마지막 질문만 던지면 ===


아직도 기억이 없네요. 지금 무슨 공부를 하고 계신지, 어떤 주제에 대해 공부하고 계신지 알려주시면 도움이 될 것 같습니다.


In [22]:
# 쌓인 대화가 실제로 어떤 입력이 되는지 확인
print('현재 messages:')
for m in chat.messages:
    content = m['content'].replace('\n', ' ')
    print(f"  {m['role']:<10} {content[:60]}")
print()
print(f'대화틀로 펼친 입력 토큰 수: {chat.token_count()}개')

현재 messages:
  system     당신은 한국어를 사용하는 친절한 AI 친구입니다.
  user       내 이름은 상은이야. 반가워!
  assistant  안녕하세요 상은야! 반가워요! 오늘은 어떻게 되세요? 어떤 일에 도움을 드릴 수 있을까요?
  user       나는 파이토치로 딥러닝을 공부하고 있어.
  assistant  파이토치로 딥러닝을 공부하고 있군요! 파이토치가 정말 유용한 도구입니다. 어떤 문제에 대해 공부하고 계신지,
  user       내가 아까 뭘 공부한다고 했지?
  assistant  저는 아마도 그 전에 다른 대화에서 파이토치로 딥러닝을 공부하는 것을 언급했을 것 같아요. 그때는 어떤 내용
  user       그럼 내 이름은 뭐였지?
  assistant  내가 पहल에 기억한 네 이름은 상은야였어요!

대화틀로 펼친 입력 토큰 수: 330개


### 풀이 해설 — 연습 문제 12-4

**핵심은 한 줄이다. 모델의 지난 답변을 `assistant` 역할로 함께 넘긴다.**

```python
self.messages.append({'role': 'user', 'content': user_text})
answer = ...
self.messages.append({'role': 'assistant', 'content': answer})
```

본문이 [코드 12-5] 아래에서 이렇게 안내한 그대로다.

> 이전 대화의 흐름을 이어 가고 싶다면 `assistant` 역할의 메시지로 모델의 과거
> 답변을 함께 넘기면 된다.

**LLM은 상태를 갖지 않는다.** 매 호출이 독립적이므로 "기억"처럼 보이는 것은
전부 **입력에 다시 실어 보낸 결과**다. 위의 대조 셀이 이를 보여 준다. 맥락 없이
"내가 아까 뭘 공부한다고 했지?"만 던지면 모델은 답할 수 없다.

**그래서 두 가지를 관리해야 한다.**

**하나, 입력이 계속 길어진다.** 누적 토큰 수 출력을 보면 턴마다 늘어난다.
콘텍스트 윈도우를 넘기면 오류가 나거나 앞부분이 잘리므로, `_trim()`처럼
**오래된 턴부터 버리는 장치**가 필요하다. 시스템 메시지는 대화의 성격을
정하므로 항상 남긴다.

**둘, 비용도 길이에 비례한다.** API 호출 방식이라면 매 턴 전체 대화를 다시
보내므로 요금이 누적된다. 실무에서는 오래된 대화를 요약해 한 메시지로 압축하는
방법을 함께 쓴다.

**적절성: 매우 좋다.** "LLM은 기억하지 않는다"는 사실을 실습으로 깨닫게 하는
문제다. 12-4절 RAG가 외부 문서를 프롬프트에 끼워 넣는 것과 **같은 원리**라,
다음 절로 가는 다리가 된다.

In [23]:
unload('model', 'tokenizer')
print(f'해제 후 GPU 메모리: {gpu_mb():.0f} MB')

해제 후 GPU 메모리: 6153 MB


---